conexión con sentinel-2 y datos


In [3]:
%pip install sentinelhub

Defaulting to user installation because normal site-packages is not writeable
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/6.3 MB 4.5 MB/s eta 0:00:02
   -------- ------------------------------- 1.3/6.3 MB 4.0 MB/s eta 0:00:02
   ----------- ---------------------------- 1.8/6.3 MB 3.2 MB/s eta 0:00:02
   -------------- ------------------------- 2.4/6.3 MB 3.0 MB/s eta 0:00:02
   ------------------- -------------------- 3.1/6.3 MB 3.0 MB/s eta 0:00:02
   --------------------- ------------------ 3.4/6.3 MB 2.7 MB/s eta 0:00:02
   --------------------- ------------------ 3.4/6.3 MB 2.7 MB/s eta 0:00:02
   ----------------------- ---------------- 3.7/6.3 MB 2.4 MB/s eta 0:00:02
   ------------------------ --------------- 3.9/6.3 MB 2.2 MB/s eta 0:00:02
   -------------------------- ------------- 4.2/6.3 MB 2.1 MB/s eta 0:00:02
   ------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%pip install rasterio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/30.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/30.6 MB ? eta -:--:--
   - -------------------------------------- 0.8/30.6 MB 2.4 MB/s eta 0:00:13
   - -------------------------------------- 1.0/30.6 MB 2.1 MB/s eta 0:00:14
   -- ------------------------------------- 1.6/30.6 MB 1.9 MB/s eta 0:00:16
   -- ------------------------------------- 1.6/30.6 MB 1.9 MB/s eta 0:00:16
   -- ------------------------------------- 1.6/30.6 MB 1.9 MB/s eta 0:00:16
   -- ------------------------------------- 1.8/30.6 MB 1.3 MB/s eta 0:00:22
   -- ------------------------------------- 2.1/30.6 MB 1.3 MB/s eta 0:00:23
   --- ------------------------------------ 2.4/30.6 MB 1.2 MB/s eta 0:00:24
   --- ------------------------------------ 2.4/30.6 MB 1.2 MB/s eta 0:00:24
   --- ------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\belen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
from datetime import datetime

from sentinelhub import (
    SHConfig,
    CRS,
    BBox,
    DataCollection,
    SentinelHubRequest,
    MimeType,
    bbox_to_dimensions
)

# para datos espaciales
import rasterio
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt


In [10]:
# vamos a usar la config de defecto por mientras
config = SHConfig()
print(f"Configuración de Sentinel Hub: {config}" )

Configuración de Sentinel Hub: {
  "instance_id": "",
  "sh_client_id": "",
  "sh_client_secret": "",
  "sh_base_url": "https://services.sentinel-hub.com",
  "sh_auth_base_url": null,
  "sh_token_url": "https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token",
  "geopedia_wms_url": "https://service.geopedia.world",
  "geopedia_rest_url": "https://www.geopedia.world/rest",
  "aws_access_key_id": "",
  "aws_secret_access_key": "",
  "aws_session_token": "",
  "aws_metadata_url": "https://roda.sentinel-hub.com",
  "aws_s3_l1c_bucket": "sentinel-s2-l1c",
  "aws_s3_l2a_bucket": "sentinel-s2-l2a",
  "opensearch_url": "http://opensearch.sentinel-hub.com/resto/api/collections/Sentinel2",
  "max_wfs_records_per_query": 100,
  "max_opensearch_records_per_query": 500,
  "max_download_attempts": 4,
  "download_sleep_time": 5.0,
  "download_timeout_seconds": 120.0,
  "number_of_download_processes": 1,
  "max_retries": null
}


#### coordenadas de los lagos

In [11]:
# coordenadas de los lagos que están en el lab 
lagos = {
    'Atitlán': {
        'west': -91.326256 ,
        'east': -91.07151 ,
        'south': 14.5948 ,
        'north': 14.750979
    },

    'Amatitlán': {
        'west': -90.638065 ,
        'east': -90.512924 ,
        'south': 14.412347 ,
        'north': 14.493799
    }
}

# fechas dadas en el lab
fechas_atitlan = [ '2025-01-18', '2025-04-13', '2025-05-13', '2025-07-17', '2025-11-21', '2025-12-29', '2026-02-12', '2026-03-24', '2026-04-13', '2026-04-28', '2026-07-22' ]

fechas_amatitlan = [ '2025-01-28', '2025-04-15', '2025-04-28', '2025-11-24', '2026-01-08', '2026-02-02', '2026-02-07', '2026-03-29', '2026-04-13', '2026-04-28', '2026-06-19' ]

fechas = { 'Atitlán': fechas_atitlan, 'Amatitlán': fechas_amatitlan }

print("Lagos definidos:")
for lago, coords in lagos.items(): print(f"  {lago}: {coords}")
print(f"\nFechas por lago:")
for lago, fecha_list in fechas.items(): print(f"  {lago}: {len(fecha_list)} fechas")

Lagos definidos:
  Atitlán: {'west': -91.326256, 'east': -91.07151, 'south': 14.5948, 'north': 14.750979}
  Amatitlán: {'west': -90.638065, 'east': -90.512924, 'south': 14.412347, 'north': 14.493799}

Fechas por lago:
  Atitlán: 11 fechas
  Amatitlán: 11 fechas


directorio para guardar datos

In [12]:
# Crear carpeta para guardar datos descargados
data_dir = Path('datos_sentinel')
data_dir.mkdir(exist_ok=True)

# Subcarpetas para cada lago
for lago in lagos.keys():
    lago_dir = data_dir / lago
    lago_dir.mkdir(exist_ok=True)

print(f"Directorio de datos: {data_dir.absolute()}")
print(f"Subdirectorios creados para cada lago")

Directorio de datos: c:\Users\belen\Documents\Data Science\DataS_Lab4_GeoEspaciales\datos_sentinel
Subdirectorios creados para cada lago


#### ddescargar datos de Sentinel-2

aquí bandas necesarias para calcular 
NDVI: B04 (Rojo) y B08 (NIR - Near Infrared)
NDWI: B03 (Verde) y B08 (NIR)
Cianobacteria